<a href="https://colab.research.google.com/github/ArjunHirani/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit


## 1. Two paper findings + my methodology questions

**Finding A — "What Predicts Health?" (Random Forest, ML Appendix, p.27).** The paper reports
`avg_position` at 43% feature importance for predicting `health_score`, followed by impressions
(32%) and scroll depth (15%). My methodology question: the paper itself discloses that
`health_score = impressions(30pts) + position(30pts) + CTR(20pts) + scroll(20pts)` — a weighted
sum where `avg_position` and `impressions` are literal input terms. That makes this close to the
skill's **taxonomy #1, label-derived features**: a feature towering in importance because it is
partially the same quantity as the label, not because it predicts an independent outcome. To the
paper's real credit, it self-discloses this ("importance is descriptive rather than causal") —
that honesty is exactly the standard I want to hold my own work to, and it's a good example of a
public paper flagging its own limitation rather than hiding it.

**Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.28).** The
methodology page confirms an 80/20 holdout split for this model but does not say whether it is
**grouped by brand** or a random row-level split, despite the study spanning 57 brands with
visible cross-brand variance elsewhere in the paper (e.g. the cluster health tables). My
methodology question: was this split grouped by brand? If it was a random row-level split, the
same brand's other content items could appear in both train and test — precisely the risk my own
Week 3-5 notebooks flagged and then fixed for my own model. This isn't a claim the finding is
wrong; it's the same honest question I'd want asked of my own 0.682-vs-0.715 gap.

In [1]:
# Grounding both questions in the paper's own disclosed numbers, not assumption.
finding_a = {
    "claim": "avg_position = 43% feature importance predicting health_score (Random Forest)",
    "disclosed_label_formula": "health_score = impressions(30pts) + position(30pts) + CTR(20pts) + scroll(20pts)",
    "taxonomy_match": "Leakage taxonomy #1: label-derived feature (position is a literal term in the label)",
    "paper_self_discloses_limit": True
}
finding_b = {
    "claim": "Logistic Regression, 71% holdout accuracy predicting growth vs decline",
    "disclosed_split": "80/20 split (methodology page) -- grouping (by brand?) not specified",
    "n_brands_in_study": 57,
    "my_question": "Was the 80/20 split grouped by brand, or random row-level across all content?"
}
for label, d in [("Finding A", finding_a), ("Finding B", finding_b)]:
    print(label + ":")
    for k, v in d.items():
        print(f"  {k}: {v}")
    print()

Finding A:
  claim: avg_position = 43% feature importance predicting health_score (Random Forest)
  disclosed_label_formula: health_score = impressions(30pts) + position(30pts) + CTR(20pts) + scroll(20pts)
  taxonomy_match: Leakage taxonomy #1: label-derived feature (position is a literal term in the label)
  paper_self_discloses_limit: True

Finding B:
  claim: Logistic Regression, 71% holdout accuracy predicting growth vs decline
  disclosed_split: 80/20 split (methodology page) -- grouping (by brand?) not specified
  n_brands_in_study: 57
  my_question: Was the 80/20 split grouped by brand, or random row-level across all content?



## 2. My model under an honest split (before/after)

Rather than just citing last week's numbers, I rebuild the comparison fresh here: the same
Random Forest, the same March→April feature set, trained and evaluated twice — once under a
**naive random row-level split** ("before"), once under the **client-grouped split** ("after") —
so the before/after gap is computed in this notebook, not remembered from a different one.

In [2]:
%pip -q install duckdb huggingface_hub
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

data = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS impressions_march,
               SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks_march,
               AVG(CASE WHEN gsc_data_available THEN gsc_avg_position END) AS avg_position_march,
               COUNT(DISTINCT CASE WHEN gsc_data_available THEN report_date END) AS active_days_march
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2
        HAVING impressions_march >= 100
    ),
    april AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks_april
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04'
        GROUP BY 1, 2
    )
    SELECT m.*, a.clicks_april FROM march m JOIN april a USING (client_hash_id, content_hash_id)
""").df()

content_meta = con.sql(f"SELECT client_hash_id, content_hash_id, content_created_date FROM {TABLES['dim_content']}").df()
data = data.merge(content_meta, on=['client_hash_id', 'content_hash_id'], how='left')
data['content_age_days_march'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(data['content_created_date'])).dt.days
data['ctr_march'] = data['clicks_march'] / data['impressions_march']
data['is_declining_april'] = (data['clicks_april'] < data['clicks_march']).astype(int)
data['position_tier'] = pd.cut(data['avg_position_march'], bins=[0, 3, 10, 20, 50, np.inf],
                                labels=['top3', '4-10', '11-20', '21-50', '50+'])
expected_ctr = data.groupby('position_tier', observed=True)['ctr_march'].transform('mean')
data['ctr_gap'] = data['ctr_march'] - expected_ctr
data = data.dropna(subset=['content_age_days_march', 'avg_position_march', 'ctr_march', 'ctr_gap']).reset_index(drop=True)

feature_cols = ['impressions_march', 'clicks_march', 'avg_position_march',
                 'active_days_march', 'content_age_days_march', 'ctr_march', 'ctr_gap']
print(f"Rows: {len(data):,}  |  distinct clients: {data['client_hash_id'].nunique()}  |  base rate: {data['is_declining_april'].mean():.3f}")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 101,441  |  distinct clients: 44  |  base rate: 0.402


In [3]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X, y = data[feature_cols], data['is_declining_april']

# --- BEFORE: naive random row-level split ---
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_before = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(Xb_tr, yb_tr)
scores_before = rf_before.predict_proba(Xb_te)[:, 1]

# --- AFTER: client-grouped split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(data, groups=data['client_hash_id']))
Xa_tr, ya_tr = data.iloc[tr_idx][feature_cols], data.iloc[tr_idx]['is_declining_april']
Xa_te, ya_te = data.iloc[te_idx][feature_cols], data.iloc[te_idx]['is_declining_april']
overlap = set(data.iloc[tr_idx]['client_hash_id']) & set(data.iloc[te_idx]['client_hash_id'])
assert len(overlap) == 0, "Client leakage across the grouped split!"
rf_after = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(Xa_tr, ya_tr)
scores_after = rf_after.predict_proba(Xa_te)[:, 1]

before_after = pd.DataFrame([
    {'split': 'BEFORE (random row-level)', 'n_test': len(yb_te),
     'precision_at_20': round(precision_at_k(scores_before, yb_te.values, 20), 3),
     'precision_at_50': round(precision_at_k(scores_before, yb_te.values, 50), 3),
     'roc_auc': round(roc_auc_score(yb_te, scores_before), 3),
     'base_rate': round(yb_te.mean(), 3)},
    {'split': 'AFTER (client-grouped)', 'n_test': len(ya_te),
     'precision_at_20': round(precision_at_k(scores_after, ya_te.values, 20), 3),
     'precision_at_50': round(precision_at_k(scores_after, ya_te.values, 50), 3),
     'roc_auc': round(roc_auc_score(ya_te, scores_after), 3),
     'base_rate': round(ya_te.mean(), 3)},
])
print("Client overlap between train/test in the grouped split (should be EMPTY):", overlap)
print()
print(before_after.to_string(index=False))

Client overlap between train/test in the grouped split (should be EMPTY): set()

                    split  n_test  precision_at_20  precision_at_50  roc_auc  base_rate
BEFORE (random row-level)   25361             0.95             0.92    0.875      0.402
   AFTER (client-grouped)    8356             0.95             0.98    0.856      0.376


## 3. Leakage audit


Repeating the Week 3 deliberate-leak trick, now on all 7 features from Week 5's final set, using
the honest client-grouped split as the base to attack.

In [4]:
from sklearn.linear_model import LogisticRegression

# Attack checklist, run explicitly:
never_features = {'clicks_april', 'is_declining_april'}
used_features = set(feature_cols)
print("[1] Timeline check -- any forbidden (future/label) column in the feature set?")
print("    Overlap (should be EMPTY):", used_features & never_features)
assert len(used_features & never_features) == 0

print("\n[2] Deliberate leak test -- add clicks_april as a feature, watch the score jump:")
Xa_tr_leaky = data.iloc[tr_idx][feature_cols + ['clicks_april']]
Xa_te_leaky = data.iloc[te_idx][feature_cols + ['clicks_april']]
leaky_model = LogisticRegression(max_iter=1000).fit(Xa_tr_leaky, ya_tr)
leaky_auc = roc_auc_score(ya_te, leaky_model.predict_proba(Xa_te_leaky)[:, 1])
honest_logreg = LogisticRegression(max_iter=1000).fit(Xa_tr, ya_tr)
honest_auc = roc_auc_score(ya_te, honest_logreg.predict_proba(Xa_te)[:, 1])
print(f"    Honest ROC-AUC (7 clean features): {honest_auc:.3f}")
print(f"    LEAKY ROC-AUC (+ clicks_april):    {leaky_auc:.3f}  <- confirms the harness catches leakage")
del Xa_tr_leaky, Xa_te_leaky, leaky_model

print("\n[3] Product-flag check -- none of FlyRank's own flags (health_score, optimization flags,")
print("    trend_direction) appear anywhere in feature_cols:", feature_cols)

print("\n[4] Population-selection check -- does the row filter depend on outcome-window info?")
pct_filtered_on_march_only = True  # HAVING impressions_march >= 100 -- filters on March, not April
print(f"    Filter is HAVING impressions_march >= 100 -- March-only, no April/outcome information used")

print("\n[5] Base rate printed next to metric -- done throughout Section 2 (base_rate column).")
print("[6] Split grouped by client -- confirmed empty overlap set in Section 2.")
print("[7] Top feature importance sanity-checked in Week 5 (ctr_march / clicks_march, not suspiciously perfect).")

[1] Timeline check -- any forbidden (future/label) column in the feature set?
    Overlap (should be EMPTY): set()

[2] Deliberate leak test -- add clicks_april as a feature, watch the score jump:
    Honest ROC-AUC (7 clean features): 0.715
    LEAKY ROC-AUC (+ clicks_april):    1.000  <- confirms the harness catches leakage

[3] Product-flag check -- none of FlyRank's own flags (health_score, optimization flags,
    trend_direction) appear anywhere in feature_cols: ['impressions_march', 'clicks_march', 'avg_position_march', 'active_days_march', 'content_age_days_march', 'ctr_march', 'ctr_gap']

[4] Population-selection check -- does the row filter depend on outcome-window info?
    Filter is HAVING impressions_march >= 100 -- March-only, no April/outcome information used

[5] Base rate printed next to metric -- done throughout Section 2 (base_rate column).
[6] Split grouped by client -- confirmed empty overlap set in Section 2.
[7] Top feature importance sanity-checked in Week 5 (ctr

## 4. Claim rewrite

**Original (Week 5, Section 3 write-up):**
> "Random Forest clearly wins on both metrics (Precision@50: 0.96 vs baseline 0.00, ROC-AUC
> 0.856), and Logistic Regression already beats the baseline comfortably too."

This is bolder than the evidence supports in two ways: "clearly wins" reads as a general,
portable claim, and it doesn't carry the caveat — raised in that same week's own error
analysis — that some of that precision may reflect a label-definition artifact (the strict `<`
threshold on low-volume March clicks), not pure decline signal.

**Rewritten:**
> "On this client-grouped test split (11 held-out clients, March→April 2026), Random Forest
> showed a measured Precision@50 of 0.96 against the baseline rule's 0.00 and Logistic
> Regression's 0.66. This is a directional result in favor of the learned model over both the
> hand rule and a simpler model on this specific split and label definition — not a claim that
> Random Forest generally outperforms simpler methods, and not fully separated from the
> low-volume label-construct concern noted in the error analysis. It supports prioritizing
> Random Forest for further validation, not a final production decision on its own."

In [5]:
claim_audit = {
    "original": "Random Forest clearly wins on both metrics...",
    "issue": "generalizes beyond one split/label definition; omits label-construct caveat from own error analysis",
    "safe_language_used": ["measured", "directional", "on this split", "decision-support, not final"],
    "forbidden_language_removed": ["clearly wins", "comfortably"]
}
for k, v in claim_audit.items():
    print(f"{k}: {v}")

original: Random Forest clearly wins on both metrics...
issue: generalizes beyond one split/label definition; omits label-construct caveat from own error analysis
safe_language_used: ['measured', 'directional', 'on this split', 'decision-support, not final']
forbidden_language_removed: ['clearly wins', 'comfortably']
